# SentinelLM — Qwen2.5-Coder-1.5B Fine-tuning
**Author:** @who_is_the_black_hat

**Model:** Qwen2.5-Coder-1.5B-Instruct + LoRA fine-tuning

**Capabilities after fine-tuning:**
- Linux/Kali environment control
- Security tool selection & chaining
- Command generation from natural language
- Autonomous scan planning
- Report generation

**Upload to Google Drive:**
- `round1_basic.jsonl`
- `round2_medium.jsonl`
- `round3_high.jsonl`

**Downloads:**
- `sentinel_lm_adapter/` (LoRA adapter ~50MB)
- `sentinel_lm_vocab.json`

```bash
# Kali pe copy karo
cp -r ~/Downloads/sentinel_lm_adapter /home/kali/osints/models/
```

In [ ]:
# Cell 1 — Install Dependencies
!pip install -q -U transformers==4.46.3
!pip install -q -U peft==0.13.2
!pip install -q -U trl==0.12.0
!pip install -q -U accelerate==0.34.2
!pip install -q -U datasets==3.1.0

import torch
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA :', torch.cuda.is_available())
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
print('PyTorch:', torch.__version__)
print('Ready!')


In [ ]:
# Cell 2 — Load & Format Training Data
from google.colab import drive
import json, glob, random
from collections import Counter

drive.mount('/content/drive', force_remount=True)

SYSTEM_PROMPT = """You are SentinelLM, an autonomous security AI running on Kali Linux.
You can:
- Understand user security objectives in any language
- Generate exact Kali Linux commands for security tools
- Plan and chain security tools (nmap → nikto → nuclei → sqlmap)
- Monitor systems and analyze findings
- Generate professional security reports
Always replace TARGET_DOMAIN with the actual target."""

def format_sample(s):
    """Convert to Qwen2.5 chat format"""
    task = s.get('task', 'cmd_gen')
    inp  = s.get('input', '').strip()
    out  = s.get('output', '').strip()

    if not inp or not out or len(out) < 2:
        return None

    # Task-specific user prompt
    if task == 'cmd_gen':
        user = f"Generate the exact Kali Linux command for:\n{inp}"
    elif task == 'chain_gen':
        user = f"What is the next security tool to use?\n{inp}"
    else:  # report_gen
        user = f"Generate a security report for these findings:\n{inp}"

    # Qwen2.5 chat template
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{out}<|im_end|>"
    )
    return {'text': text, 'task': task}

def load_jsonl(path, max_samples=25000):
    samples = []
    seen = set()
    with open(path, encoding='utf-8', errors='replace') as f:
        for line in f:
            if len(samples) >= max_samples: break
            try:
                d = json.loads(line.strip())
                key = d.get('input','')[:60] + d.get('output','')[:30]
                if key in seen: continue
                seen.add(key)
                fmt = format_sample(d)
                if fmt: samples.append(fmt)
            except: pass
    return samples

all_samples = []
for fname in ['round1_basic.jsonl', 'round2_medium.jsonl', 'round3_high.jsonl']:
    found = glob.glob(f'/content/drive/MyDrive/**/{fname}', recursive=True)
    if not found: found = glob.glob(f'/content/drive/MyDrive/{fname}')
    if found:
        s = load_jsonl(found[0])
        all_samples.extend(s)
        print(f'✓ {fname}: {len(s):,} samples')
    else:
        print(f'✗ {fname}: NOT FOUND')

random.shuffle(all_samples)
print(f'\nTotal: {len(all_samples):,}')
print(f'Tasks: {dict(Counter(s["task"] for s in all_samples))}')
print(f'\nSample:')
print(all_samples[0]['text'][:300])


In [ ]:
# Cell 3 — Load Qwen2.5-Coder-1.5B + LoRA (float16, no bitsandbytes)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Loading {MODEL_ID} on {DEVICE}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# float16 — T4 pe 15.6GB VRAM hai, 1.5B model fit hoga
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Base model loaded | {total_params:.0f}M params')

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'LoRA applied | Device: {DEVICE}')


In [ ]:
# Cell 4 — Fine-tune with SFTTrainer
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import time, random

MAX_SEQ_LEN = 512

n_val = max(200, int(len(all_samples) * 0.05))
random.shuffle(all_samples)
tr_data = Dataset.from_list(all_samples[n_val:])
vl_data = Dataset.from_list(all_samples[:n_val])
print(f'Train: {len(tr_data):,} | Val: {len(vl_data):,}')

training_args = SFTConfig(
    output_dir='/content/sentinel_lm_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    optim='adamw_torch',
    weight_decay=0.01,
    max_grad_norm=0.3,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tr_data,
    eval_dataset=vl_data,
    tokenizer=tokenizer,
)

print('Training started...')
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f'\nTraining done in {elapsed/60:.1f} min')
print(f'Best eval loss: {trainer.state.best_metric}')


In [ ]:
# Cell 5 — Inference Test
from transformers import pipeline

def generate(user_prompt, max_new_tokens=150, temperature=0.3):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

print('=== SentinelLM Inference Test ===\n')
tests = [
    # Linux basics
    'list all files recursively in current directory',
    'find all python files modified in last 7 days',
    # Security tools
    'scan example.com for open ports and services',
    'check example.com for sql injection vulnerabilities',
    'enumerate subdomains of example.com',
    'scan example.com for wordpress vulnerabilities',
    'brute force ssh login on example.com',
    # Chain
    'nmap found port 80 open on example.com, what next?',
    'nikto found xss vulnerability, what tool to use next?',
    # Report
    'generate report: sql injection found on login form, severity critical',
    # Urdu/Hindi
    'example.com ke ports scan karo',
    'wordpress vulnerabilities dhundo example.com mein',
]
for prompt in tests:
    out = generate(prompt)
    print(f'User : {prompt}')
    print(f'Model: {out[:120]}')
    print()


In [ ]:
# Cell 6 — Save LoRA Adapter + Download
import os, shutil, json, time
from google.colab import files

SAVE_DIR = '/content/sentinel_lm_adapter'

# Save LoRA adapter only (~50MB instead of full 3GB)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save metadata
meta = {
    'base_model':    'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    'adapter_type':  'LoRA',
    'lora_r':        16,
    'lora_alpha':    32,
    'tasks':         ['cmd_gen', 'chain_gen', 'report_gen'],
    'system_prompt': SYSTEM_PROMPT,
    'train_samples': len(tr_data),
    'best_eval_loss': trainer.state.best_metric,
    'author':        'who_is_the_black_hat',
    'version':       '1.0',
    'saved_at':      time.strftime('%Y-%m-%d %H:%M:%S'),
}
json.dump(meta, open(f'{SAVE_DIR}/sentinel_meta.json', 'w'), indent=2)

# Zip karo
shutil.make_archive('/content/sentinel_lm_adapter', 'zip', '/content', 'sentinel_lm_adapter')

print(f'Adapter size: {os.path.getsize("/content/sentinel_lm_adapter.zip")/1e6:.1f} MB')
print(f'Train samples: {len(tr_data):,}')
print(f'Best eval loss: {trainer.state.best_metric}')
print()

files.download('/content/sentinel_lm_adapter.zip')
print('Downloaded!')
print()
print('Kali pe copy karo:')
print('unzip ~/Downloads/sentinel_lm_adapter.zip -d /home/kali/osints/models/')
